![DB Academy](https://files.training.databricks.com/binder/prod_main/get-started-with-databricks-for-data-warehousing-en_us-3.0.3/images/20261012T131742Z/Get Started with Databricks for Data Warehousing/Includes/images/db-academy.png)

<div style="max-width: 1000px; margin: 0 auto; font-family: sans-serif;">
<div style="background: #1B5162; color: white; border-radius: 8px; padding: 28px 32px; text-align: center; position: relative;">
  <div style="font-size: 14pt; font-weight: 600; text-transform: uppercase; letter-spacing: 1px; opacity: 0.85; margin-bottom: 8px;">Lesson 06</div>
  <div style="font-size: 24pt; font-weight: 700; line-height: 1.3;">Loading Data into the Warehouse</div>
  <div style="font-size: 14pt; margin-top: 12px; opacity: 0.9;">Use Auto Loader to incrementally ingest files into a Unity Catalog managed table, with exactly-once safety on every re-run.</div>
</div>
</div>

## REQUIRED: SELECT A COMPUTE ENVIRONMENT

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 14px 18px; border-radius: 4px; margin: 16px 0;">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Select Serverless Compute</strong>
  <div style="color:#333;">

This notebook runs on Serverless compute. Before running any cells, confirm Serverless is selected in the compute dropdown at the top-right of this notebook. This course was developed and tested on <strong>Serverless environment version 5</strong>.
  </div>
</div>

### Setup
Run the cell below to configure your environment for this lesson. The setup script initializes your catalog and schema so your queries run against the correct context. It does **not** load any files yet. We will load them together later in this lesson, after you have seen how Auto Loader works.


In [0]:
%run ./Includes/Classroom-Setup-06

## What this lesson covers

The setup staged six quarterly order CSVs in your `data` volume but didn't load them. This lesson loads the four 2024 files into a UC managed table called `orders_bronze` using **Auto Loader** - Databricks' flagship tool for ingesting files from cloud storage.

You'll: see the source files, learn what Auto Loader is and why its re-runs are safe, run a narrated load, verify the result, and see where Auto Loader fits among the other ingestion options.

## 1. See the source files

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 14px 18px; border-radius: 4px; margin: 12px 0; font-size: 15px; color: #333; line-height: 1.6;">
  <strong style="color: #1565c0; font-size: 16px;">Try this in Catalog Explorer</strong>
  <ol style="margin: 8px 0 0 0; padding-left: 20px;">
    <li>In the left sidebar, click <strong>Catalog</strong>.</li>
    <li>Navigate to your <code style="font-size: 15px;">labuser...</code> catalog &#x2192; <strong>vintage_audio</strong> &#x2192; <strong>Volumes</strong> &#x2192; <strong>data</strong>.</li>
    <li>You should see the four 2024 files (<code style="font-size: 15px;">orders_2024_q1.csv</code> through <code style="font-size: 15px;">q4.csv</code>) and two 2025 files (used in Lesson 07).</li>
  </ol>
</div>

## 2. What Auto Loader is

Auto Loader incrementally and efficiently processes new data files as they arrive in cloud storage, without any additional setup. It provides a Structured Streaming source called **`cloudFiles`**: you point it at a directory, and it automatically detects and ingests new files as they land. It **tracks which files it has already processed in a checkpoint**, so each file is ingested **exactly once**, no matter how many times the stream runs, and it scales to billions of files and near-real-time ingestion of millions of files per hour.

<div style="width: 100%; margin: 16px 0;">
<svg viewBox="0 0 720 320" xmlns="http://www.w3.org/2000/svg" font-family="sans-serif" style="width: 100%; height: auto; display: block;">
  <text x="360" y="32" text-anchor="middle" fill="#1B5162" font-size="19" font-weight="700">HOW AUTO LOADER INGESTS FILES</text>
  <!-- Source volume -->
  <rect x="20" y="70" width="190" height="180" rx="10" fill="#F9F7F4" stroke="#1B5162" stroke-width="1.5"/>
  <text x="115" y="96" text-anchor="middle" fill="#1B5162" font-size="16" font-weight="700">data volume</text>
  <rect x="40" y="112" width="150" height="30" rx="4" fill="#FFFFFF" stroke="#FE3722" stroke-width="1.3"/>
  <text x="115" y="132" text-anchor="middle" fill="#0b2026" font-size="15">orders_2024_q1.csv</text>
  <rect x="40" y="148" width="150" height="30" rx="4" fill="#FFFFFF" stroke="#FE3722" stroke-width="1.3"/>
  <text x="115" y="168" text-anchor="middle" fill="#0b2026" font-size="15">orders_2024_q2.csv</text>
  <rect x="40" y="184" width="150" height="30" rx="4" fill="#FFFFFF" stroke="#FE3722" stroke-width="1.3"/>
  <text x="115" y="204" text-anchor="middle" fill="#0b2026" font-size="15">orders_2024_q3.csv</text>
  <rect x="40" y="220" width="150" height="20" rx="4" fill="#FFFFFF" stroke="#B5BFC4" stroke-width="1" stroke-dasharray="3,2"/>
  <text x="115" y="234" text-anchor="middle" fill="#5E7077" font-size="15">&hellip; new files later</text>
  <!-- Auto Loader engine -->
  <rect x="265" y="120" width="190" height="80" rx="10" fill="#1B5162"/>
  <text x="360" y="150" text-anchor="middle" fill="#FFFFFF" font-size="16" font-weight="700">Auto Loader</text>
  <text x="360" y="174" text-anchor="middle" fill="#FFFFFF" font-size="15">cloudFiles source</text>
  <!-- Target table -->
  <rect x="510" y="120" width="190" height="80" rx="10" fill="#F9F7F4" stroke="#00A972" stroke-width="1.8"/>
  <text x="605" y="150" text-anchor="middle" fill="#0b2026" font-size="16" font-weight="700">orders_bronze</text>
  <text x="605" y="174" text-anchor="middle" fill="#5E7077" font-size="15">UC managed table</text>
  <!-- Arrows -->
  <line x1="210" y1="160" x2="263" y2="160" stroke="#5E7077" stroke-width="2"/>
  <polygon points="263,160 253,155 253,165" fill="#5E7077"/>
  <line x1="455" y1="160" x2="508" y2="160" stroke="#5E7077" stroke-width="2"/>
  <polygon points="508,160 498,155 498,165" fill="#5E7077"/>
  <!-- Checkpoint -->
  <rect x="265" y="232" width="190" height="56" rx="8" fill="#FFF7EC" stroke="#FF9800" stroke-width="1.5"/>
  <text x="360" y="256" text-anchor="middle" fill="#FF9800" font-size="15" font-weight="700">checkpoint</text>
  <text x="360" y="278" text-anchor="middle" fill="#0b2026" font-size="15">remembers processed files</text>
  <line x1="360" y1="200" x2="360" y2="230" stroke="#FF9800" stroke-width="1.6" stroke-dasharray="4,3"/>
</svg>
</div>

## 3. Why re-runs are safe

Because the checkpoint records every processed file, running the same Auto Loader stream twice ingests each file **exactly once**. The second run finds nothing new and ingests zero rows. The narrated load in Section 4 shows this directly.

<div style="width: 100%; margin: 16px 0;">
<svg viewBox="0 0 720 340" xmlns="http://www.w3.org/2000/svg" font-family="sans-serif" style="width: 100%; height: auto; display: block;">
  <text x="360" y="30" text-anchor="middle" fill="#1B5162" font-size="19" font-weight="700">EXACTLY-ONCE ON EVERY RE-RUN</text>
  <!-- First run -->
  <rect x="30" y="54" width="300" height="230" rx="10" fill="#F9F7F4" stroke="#1B5162" stroke-width="1.5"/>
  <text x="180" y="80" text-anchor="middle" fill="#0b2026" font-size="16" font-weight="700">First run</text>
  <text x="180" y="102" text-anchor="middle" fill="#5E7077" font-size="15">4 files in the volume, none seen yet</text>
  <rect x="55" y="114" width="250" height="100" rx="6" fill="#FFFFFF" stroke="#FE3722" stroke-width="1.5"/>
  <text x="90" y="138" fill="#5E7077" font-size="14">Q1:</text><text x="130" y="138" fill="#0b2026" font-size="14" font-weight="600">43,188</text>
  <text x="200" y="138" fill="#5E7077" font-size="14">Q2:</text><text x="240" y="138" fill="#0b2026" font-size="14" font-weight="600">47,507</text>
  <text x="90" y="158" fill="#5E7077" font-size="14">Q3:</text><text x="130" y="158" fill="#0b2026" font-size="14" font-weight="600">51,826</text>
  <text x="200" y="158" fill="#5E7077" font-size="14">Q4:</text><text x="240" y="158" fill="#0b2026" font-size="14" font-weight="600">60,463</text>
  <line x1="70" y1="172" x2="290" y2="172" stroke="#FE3722" stroke-width="1"/>
  <text x="180" y="196" text-anchor="middle" fill="#FE3722" font-size="16" font-weight="700">= 202,984 rows</text>
  <text x="180" y="244" text-anchor="middle" fill="#5E7077" font-size="15">checkpoint now lists all 4 files</text>
  <text x="180" y="266" text-anchor="middle" fill="#5E7077" font-size="15">as processed</text>
  <!-- Arrow -->
  <line x1="335" y1="170" x2="385" y2="170" stroke="#5E7077" stroke-width="2"/>
  <polygon points="385,170 375,165 375,175" fill="#5E7077"/>
  <!-- Re-run -->
  <rect x="390" y="54" width="300" height="230" rx="10" fill="#F9F7F4" stroke="#1B5162" stroke-width="1.5"/>
  <text x="540" y="80" text-anchor="middle" fill="#0b2026" font-size="16" font-weight="700">Re-run (same stream)</text>
  <text x="540" y="102" text-anchor="middle" fill="#5E7077" font-size="15">checkpoint checks each file:</text>
  <rect x="415" y="114" width="250" height="100" rx="6" fill="#FFFFFF" stroke="#618794" stroke-width="1.5"/>
  <text x="440" y="138" fill="#5E7077" font-size="14">Q1: </text><text x="475" y="138" fill="#B5BFC4" font-size="14" font-weight="600" text-decoration="line-through">43,188</text><text x="545" y="138" fill="#618794" font-size="14">already seen</text>
  <text x="440" y="158" fill="#5E7077" font-size="14">Q2: </text><text x="475" y="158" fill="#B5BFC4" font-size="14" font-weight="600" text-decoration="line-through">47,507</text><text x="545" y="158" fill="#618794" font-size="14">already seen</text>
  <text x="440" y="178" fill="#5E7077" font-size="14">Q3: </text><text x="475" y="178" fill="#B5BFC4" font-size="14" font-weight="600" text-decoration="line-through">51,826</text><text x="545" y="178" fill="#618794" font-size="14">already seen</text>
  <text x="440" y="198" fill="#5E7077" font-size="14">Q4: </text><text x="475" y="198" fill="#B5BFC4" font-size="14" font-weight="600" text-decoration="line-through">60,463</text><text x="545" y="198" fill="#618794" font-size="14">already seen</text>
  <text x="540" y="240" text-anchor="middle" fill="#618794" font-size="16" font-weight="700">= 0 new rows ingested</text>
  <text x="540" y="266" text-anchor="middle" fill="#5E7077" font-size="15">nothing is reprocessed</text>
  <rect x="30" y="298" width="660" height="26" rx="6" fill="#FFF7EC" stroke="#FF9800" stroke-width="1.4"/>
  <text x="360" y="316" text-anchor="middle" fill="#0b2026" font-size="15">Safe to run on any schedule -- only genuinely new files are ever ingested.</text>
</svg>
</div>

## 4. Run the load

The cell below runs the Auto Loader load while narrating each step. Watch the output: it ingests the four 2024 files, then runs the same stream again to show the re-run ingests zero new rows.

In [0]:
%run ./Includes/load-orders-bronze

## 5. Verify what was loaded

### Total row count

In [0]:
%sql
SELECT COUNT(*) AS row_count FROM orders_bronze;

Expect **202,984** rows (43,188 + 47,507 + 51,826 + 60,463). The re-run in Section 4 added nothing, confirming exactly-once ingestion.

## 6. Where Auto Loader fits

Auto Loader is the recommended tool for ingesting **files from cloud storage**, from a handful up to millions per hour. Other sources call for other tools. Start with your data source, then follow the path:

<div style="width: 100%; margin: 16px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;">
  <div style="display: grid; grid-template-columns: auto 1fr 1fr 1fr; gap: 10px 14px; align-items: stretch; font-size: 15px;">
    <!-- Header row -->
    <div style="font-weight: 700; color: #1B5162; padding: 10px 0; font-size: 16px; display: flex; align-items: center;">Your data source</div>
    <div style="font-weight: 700; color: #1B5162; padding: 10px 0; font-size: 16px; text-align: center; display: flex; align-items: center; justify-content: center;">Recommended tool</div>
    <div style="font-weight: 700; color: #5E7077; padding: 10px 0; font-size: 16px; text-align: center; display: flex; align-items: center; justify-content: center;">Scale</div>
    <div style="font-weight: 700; color: #5E7077; padding: 10px 0; font-size: 16px; text-align: center; display: flex; align-items: center; justify-content: center;">Pattern</div>
    <!-- Row 1: Auto Loader -->
    <div style="background: #F0FAF5; border: 2px solid #00A972; border-radius: 8px; padding: 10px 14px; display: flex; align-items: center; gap: 8px;">
      <span style="font-weight: 700; color: #1B3139;">Files in cloud storage</span>
      <span style="font-size: 14px; color: #5E7077;">(CSV, JSON, Parquet)</span>
    </div>
    <div style="background: #00A972; color: #fff; border-radius: 8px; padding: 10px 14px; text-align: center; font-weight: 700; font-size: 16px; display: flex; align-items: center; justify-content: center; gap: 8px;">Auto Loader <span style="font-size: 14px; font-weight: 400; opacity: 0.85;">&#x2605; this lesson</span></div>
    <div style="background: #f8fafc; border: 1.5px solid #DCE0E2; border-radius: 8px; padding: 10px 14px; text-align: center; color: #3d5a65; display: flex; align-items: center; justify-content: center;">Millions of files/hour</div>
    <div style="background: #f8fafc; border: 1.5px solid #DCE0E2; border-radius: 8px; padding: 10px 14px; text-align: center; color: #3d5a65; display: flex; align-items: center; justify-content: center;">Ongoing, incremental</div>
    <!-- Row 2: COPY INTO -->
    <div style="background: #f8fafc; border: 1.5px solid #DCE0E2; border-radius: 8px; padding: 10px 14px; display: flex; align-items: center; gap: 8px;">
      <span style="font-weight: 700; color: #1B3139;">Files in cloud storage</span>
      <span style="font-size: 14px; color: #5E7077;">(same formats)</span>
    </div>
    <div style="background: #5E7077; color: #fff; border-radius: 8px; padding: 10px 14px; text-align: center; font-weight: 700; font-size: 16px; display: flex; align-items: center; justify-content: center;"><code style="font-size: 16px; color: #fff; background: none;">COPY INTO</code></div>
    <div style="background: #f8fafc; border: 1.5px solid #DCE0E2; border-radius: 8px; padding: 10px 14px; text-align: center; color: #3d5a65; display: flex; align-items: center; justify-content: center;">Small to moderate</div>
    <div style="background: #f8fafc; border: 1.5px solid #DCE0E2; border-radius: 8px; padding: 10px 14px; text-align: center; color: #3d5a65; display: flex; align-items: center; justify-content: center;">Simple, one-off bulk loads</div>
    <!-- Row 3: Lakeflow Connect -->
    <div style="background: #f8fafc; border: 1.5px solid #DCE0E2; border-radius: 8px; padding: 10px 14px; display: flex; align-items: center; gap: 8px;">
      <span><span style="font-weight: 700; color: #1B3139;">SaaS apps and databases</span> <span style="font-size: 14px; color: #5E7077;">(Salesforce, SQL Server, etc.)</span></span>
    </div>
    <div style="background: #1B5162; color: #fff; border-radius: 8px; padding: 10px 14px; text-align: center; font-weight: 700; font-size: 16px; display: flex; align-items: center; justify-content: center;">Lakeflow Connect</div>
    <div style="background: #f8fafc; border: 1.5px solid #DCE0E2; border-radius: 8px; padding: 10px 14px; text-align: center; color: #3d5a65; display: flex; align-items: center; justify-content: center;">Managed by Databricks</div>
    <div style="background: #f8fafc; border: 1.5px solid #DCE0E2; border-radius: 8px; padding: 10px 14px; text-align: center; color: #3d5a65; display: flex; align-items: center; justify-content: center;">Ongoing sync</div>
    <!-- Row 4: Structured Streaming -->
    <div style="background: #f8fafc; border: 1.5px solid #DCE0E2; border-radius: 8px; padding: 10px 14px; display: flex; align-items: center; gap: 8px;">
      <span><span style="font-weight: 700; color: #1B3139;">Real-time event streams</span> <span style="font-size: 14px; color: #5E7077;">(Kafka, Kinesis, Event Hubs)</span></span>
    </div>
    <div style="background: #FF8C42; color: #fff; border-radius: 8px; padding: 10px 14px; text-align: center; font-weight: 700; font-size: 16px; display: flex; align-items: center; justify-content: center;">Structured Streaming</div>
    <div style="background: #f8fafc; border: 1.5px solid #DCE0E2; border-radius: 8px; padding: 10px 14px; text-align: center; color: #3d5a65; display: flex; align-items: center; justify-content: center;">Millions of events/sec</div>
    <div style="background: #f8fafc; border: 1.5px solid #DCE0E2; border-radius: 8px; padding: 10px 14px; text-align: center; color: #3d5a65; display: flex; align-items: center; justify-content: center;">Continuous / micro-batch</div>
  </div>
</div>

**Real-time ingestion** is a major theme on Databricks: Auto Loader and Structured Streaming both support near-real-time pipelines that land data within seconds of arrival, which is increasingly how analytics and AI workloads expect to consume data. The Data Engineering course goes deep on Streaming and Apache Spark Declarative Pipelines.

> **About `COPY INTO`.** `COPY INTO` is a SQL command that also loads files idempotently and is fine for simple, occasional bulk loads. For ongoing and large-scale ingestion, Databricks steers you toward Auto Loader, which discovers files more efficiently and scales much further. Treat `COPY INTO` as the lightweight SQL option, and Auto Loader as the default.

## What you learned in this lesson

<div style="border-left: 4px solid #4caf50; background: #e8f5e9; padding: 14px 18px; border-radius: 4px; margin: 16px 0;">
<div style="color:#333;">

| Concept | Key idea |
|---|---|
| Auto Loader | The `cloudFiles` streaming source that ingests files from cloud storage |
| Exactly-once | A checkpoint tracks processed files, so re-runs ingest zero duplicates |
| One-shot batch | `trigger(availableNow=True)` processes everything available, then stops |
| Verification | Confirm the total row count after loading |
| The bigger picture | Lakeflow Connect for SaaS/DB sources; Structured Streaming for real-time events; `COPY INTO` as the lightweight SQL alternative |

</div>
</div>


&copy; <span id="dbx-year"></span> Databricks, Inc. All rights reserved.
Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
<script>
  document.getElementById("dbx-year").textContent = new Date().getFullYear();
</script>